# Mapeamento do Universo Negociável de Ações na B3

Os arquivos de Séries Históricas da B3 contêm o registro diário consolidado de todas as negociações realizadas na bolsa, trazendo informações como tipo de ativo, data do pregão, código de negociação (ticker), modalidade de mercado, preços e volume financeiro. Para isolar o universo de ações investíveis, aplicamos filtros que selecionam apenas o mercado à vista e fracionário (BDIs 02 e 12, mercados 10 e 20), restringindo os tickers aos finais ordinários, preferenciais e units (3, 4, 5, 6 e 11) e descartando FIIs e ETFs pelas colunas de especificação e nome. Em seguida, aplicamos um critério de liquidez mínima de pelo menos 20 dias negociados no ano e volume financeiro acumulado de no mínimo R$ 100.000,00 para eliminar papéis "fantasmas" ou ilíquidos. Dessa forma, estruturamos uma coletânea precisa de ações efetivamente negociáveis por período, garantindo uma base sólida e realista para simulações e backtests.

## Estrutura e Premissas do Layout B3

- Tipo de Registro (00-02) | Filtro: 01
    - O que é: Identifica o conteúdo da linha. O código 01 representa a cotação histórica diária.
    - Por que: Arquivos da B3 contêm metadados (00 e 99) e resumos estatísticos que não são preços. Usar apenas o 01 garante que você está lidando com dados brutos de negociação.

- Data da Transação (02-10) | Formato: AAAAMMDD
    - O que é: O dia exato em que aqueles preços e volumes ocorreram.
    - Por que: Permite validar a frequência de negociação. Ações com presença em pregão inferior ao limiar estipulado (ex: 20 dias/ano) são descartadas para evitar papéis ilíquidos.

- Código BDI (10-12) | Filtro: 02 e 12
    - O que é: Classifica o tipo de papel (Ações, FIIs, BDRs, etc.). O código 02 representa o Lote Padrão e o 12 o Mercado Fracionário.
    - Por que: Isola o mercado acionário principal. Como FIIs antigos e ETFs também podem surgir nestes BDIs, este filtro atua em conjunto com as colunas de especificação (ESPECI) e nome (NOMRES) para garantir a captura exclusiva de ações de empresas.

- Ticker do Ativo (12-24) | Identificador
    - O que é: O código da ação (ex: VALE3, PETR4). Ocupa 12 espaços para prever códigos longos, mas no Brasil usa 5 ou 6 caracteres seguidos de espaços.
    - Por que: É a chave primária para agregação dos dados e extração do radical da empresa (primeiros 4 caracteres), permitindo identificar tickers ativos e consolidar por companhia aberta.

- Tipo de Mercado (24-27) | Filtro: 010 e 020
    - O que é: Indica a modalidade de negociação, sendo 010 o Mercado à Vista e 020 o Mercado Fracionário.
    - Por que: Remove operações com derivativos, opções e termo (070, 080), focando a análise no mercado acionário à vista.

- Nome Resumido e Especificação (27-49) | Filtro por Regex
    - O que é: NOMRES (27–39) traz o nome da empresa e ESPECI (39–49) a classe do papel (ON, PN, CI, etc.).
    - Por que usar: Filtro essencial para remover definitivamente FIIs, FIAGROs, FIPs e ETFs que compartilham os mesmos BDIs e sufixos com ações de empresas.

- Preço de Fechamento (108-121) | Cálculo: Valor / 100
    - O que é: O último preço negociado no dia. Vem sem vírgula (número inteiro com 2 casas decimais implícitas - ex: 1235 significa 12,35).
    - Por que: Serve como preço de referência para apuração de valor de mercado e cálculo de indicadores de valuation.

- Quantidade de Negócios (147-152) | Validação
    - O que é: Total de operações de compra e venda realizadas para aquele ativo no dia.
    - Por que: Permite atestar a distribuição da liquidez diária, evitando que um único grande negócio pontual mascare um ativo ilíquido.

- Volume Financeiro Total (170-188) | Cálculo: Valor / 100
    - O que é: Montante financeiro total em Reais negociado no dia (número inteiro com 2 casas decimais implícitas).
    - Por que: Utilizado no corte de liquidez mínima acumulada no ano (ex: >= R$ 100.000,00), garantindo que a base final contenha apenas ativos com liquidez simulável.


In [21]:
import pandas as pd
import os
import json

In [22]:
# Definições do Layout B3 (Largura Fixa)
COL_SPECS = [
    (0, 2),      # REG
    (2, 10),     # DATA
    (10, 12),    # BDI
    (12, 24),    # TICKER
    (24, 27),    # MERCADO
    (27, 39),    # NOMRES
    (39, 49),    # ESPECI
    (108, 121),  # PRECO
    (152, 170),  # QTD
    (170, 188),  # VOLUME
    (230, 242)   # ISIN
]

NAMES = [
    'REG',
    'DATA',
    'BDI',
    'TICKER',
    'MERCADO',
    'NOMRES',
    'ESPECI',
    'PRECO',
    'QTD',
    'VOLUME',
    'ISIN'
]


In [23]:
def processar_ano_apenas_acoes_empresas(caminho_arquivo):
    nome_arquivo = os.path.basename(caminho_arquivo)
    ano = "".join(filter(str.isdigit, nome_arquivo))
    
    # 1. Leitura
    df = pd.read_fwf(caminho_arquivo, colspecs=COL_SPECS, names=NAMES, skipfooter=1, encoding='latin1')
    
    # 2. Filtro de Registro e Mercado
    df = df[df['REG'] == 1].copy()
    df = df[df['BDI'].isin([2, 12])].copy()
    df = df[df['MERCADO'].isin([10, 20])].copy()

    # 3. Limpeza de Ticker
    df['TICKER'] = df['TICKER'].str.strip()
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    pattern = r'^[A-Z]{4}(3|4|5|6|11)$'
    df = df[df['TICKER'].str.match(pattern, na=False)].copy()

    # --- FILTRO DEFINITIVO: REMOVER FIIS PELA COLUNA ESPECI E NOME ---
    # Limpamos espaços das colunas de texto
    df['ESPECI'] = df['ESPECI'].astype(str).str.strip().str.upper()
    df['NOMRES'] = df['NOMRES'].astype(str).str.strip().str.upper()

    # Remove qualquer coisa que tenha FII, FIP, FUNDO ou CI (Cota de Indústria/Fundo) na especificação ou nome
    termo_fundo = r'FII|FIP|FUNDO|CI|FIE'
    mask_fundos = df['ESPECI'].str.contains(termo_fundo, regex=True, na=False) | \
                  df['NOMRES'].str.contains(termo_fundo, regex=True, na=False)
    
    # Ficamos APENAS com o que NÃO é fundo
    df = df[~mask_fundos].copy()

    # 4. Ajuste de volume e liquidez mínima
    df['VOLUME'] = df['VOLUME'] / 100
    
    stats = (df.groupby('TICKER').agg(dias_negociados=('DATA', 'count'),volume_total=('VOLUME', 'sum'),codigo_isin=('ISIN', 'first')).reset_index())
    
    # Filtro de liquidez mínima para simulações
    acoes_ativas = stats[(stats['dias_negociados'] >= 20) & (stats['volume_total'] >= 100000)].copy()
    
    acoes_ativas['ano'] = ano
    acoes_ativas['RADICAL'] = acoes_ativas['TICKER'].str[:4]
    
    total_tickers = acoes_ativas['TICKER'].nunique()
    total_empresas = acoes_ativas['RADICAL'].nunique()
    
    print(f"--- Processando: {nome_arquivo} ---")
    print(f"Tickers de AÇÕES REAIS: {total_tickers}")
    print(f"EMPRESAS distintas: {total_empresas}")
    print("-" * 40)
    
    return acoes_ativas


In [24]:
# --- EXECUÇÃO DO LOOP DOS ARQUIVOS ---

lista_frames = []
diretorio = "data/raw/historical_1_series_b3_extracted"
for arq in os.listdir(diretorio):
    if arq.endswith(".TXT"):
        caminho_completo = os.path.join(diretorio, arq)
        df_ano = processar_ano_apenas_acoes_empresas(caminho_completo)
        lista_frames.append(df_ano)

# Concatenar todos os anos em um único DataFrame
df_final = pd.concat(lista_frames, ignore_index=True)
df_final = df_final.sort_values(by=['ano', 'TICKER'], ascending=[True, True])

# TABELA DE EQUIVALÊNCIA DOS TICKERS

caminho_mapping = "data/interim/ticker_and_code_cvm_mapping.csv"
if os.path.exists(caminho_mapping):
    print("\nTabela de equivalência já existe.")
    print("Nenhuma alteração foi realizada em ticker_and_code_cvm_mapping.csv.")
else:
    ticker_mapping = (
        df_final
        .groupby("TICKER", as_index=False)
        .agg(
            codigo_isin=("codigo_isin", "first"),
            ano_inicial=("ano", "min"),
            ano_final=("ano", "max")
        )
    )

    # Campo que será preenchido manualmente
    ticker_mapping.insert(1, "codigo_cvm", "")

    ticker_mapping = ticker_mapping[
        [
            "TICKER",
            "codigo_cvm",
            "codigo_isin",
            "ano_inicial",
            "ano_final"
        ]
    ]

    ticker_mapping.to_csv(
        caminho_mapping,
        sep=";",
        decimal=",",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nTabela mestre criada com {len(ticker_mapping)} tickers.")

# --- GERAÇÃO DAS LISTAS COMPLETAS POR ANO (JSON) ---
tickers_por_ano = {}
for ano, grupo in df_final.groupby('ano'):
    tickers_por_ano[str(ano)] = grupo['TICKER'].unique().tolist()
with open('data/interim/investable_stocks_per_year.json', 'w') as f:
    json.dump(tickers_por_ano, f, indent=4)
print('\nCriação de arquivo json com ações investíveis por ano.')


--- Processando: COTAHIST_A2010.TXT ---
Tickers de AÇÕES REAIS: 432
EMPRESAS distintas: 324
----------------------------------------
--- Processando: COTAHIST_A2011.TXT ---
Tickers de AÇÕES REAIS: 424
EMPRESAS distintas: 320
----------------------------------------
--- Processando: COTAHIST_A2012.TXT ---
Tickers de AÇÕES REAIS: 396
EMPRESAS distintas: 308
----------------------------------------
--- Processando: COTAHIST_A2013.TXT ---
Tickers de AÇÕES REAIS: 376
EMPRESAS distintas: 298
----------------------------------------
--- Processando: COTAHIST_A2014.TXT ---
Tickers de AÇÕES REAIS: 354
EMPRESAS distintas: 282
----------------------------------------
--- Processando: COTAHIST_A2015.TXT ---
Tickers de AÇÕES REAIS: 339
EMPRESAS distintas: 273
----------------------------------------
--- Processando: COTAHIST_A2016.TXT ---
Tickers de AÇÕES REAIS: 341
EMPRESAS distintas: 269
----------------------------------------
--- Processando: COTAHIST_A2017.TXT ---
Tickers de AÇÕES REAIS: 366
E

Após identificar quais tickers foram negociados na B3 em cada ano, o próximo passo consiste em estabelecer a correspondência entre cada ticker e a companhia emissora.

Embora a CVM disponibilize o código CVM, o CNPJ e a razão social das companhias abertas, os arquivos históricos da B3 (COTAHIST), utilizados neste trabalho para identificar os ativos negociados, contêm apenas informações como o ticker e o código ISIN, não existindo uma relação direta entre um elemento para identificar a companhia (código cvm ou cnpj) e seu ticker.

Além disso, essa correspondência não pode ser considerada estática ao longo do tempo. Durante o período analisado (2010-2025), diversas companhias passaram por eventos societários, como mudanças de ticker, incorporações, fusões, reorganizações societárias e alterações na estrutura do capital. Dessa forma, um mesmo ticker pode representar empresas diferentes em momentos distintos, assim como uma mesma companhia pode negociar sob tickers diferentes ao longo dos anos.

Por esse motivo, foi construída uma tabela de equivalência (ticker_and_code_cvm_mapping.csv) contendo, para cada ticker identificado nos arquivos históricos da B3, o respectivo código CVM e o intervalo de anos em que aquela associação é válida.

Os campos ticker, codigo ISIN, ano inicial e ano final são gerados automaticamente a partir dos arquivos históricos COTAHIST, percorrendo todos os anos disponíveis da base de dados.
O único campo preenchido manualmente é o código CVM. Esse preenchimento foi realizado por meio de consulta ao cadastro público de companhias abertas da CVM, disponível em:
https://cvmweb.cvm.gov.br/SWB/Sistemas/SCW/CPublica/CiaAb/FormBuscaCiaAb.aspx?TipoConsult=c

Essa etapa manual foi escolhida por oferecer maior confiabilidade do que a tentativa de estabelecer automaticamente a correspondência entre diferentes bases cadastrais da B3 e da CVM, além de envolver um número relativamente reduzido de registros (aproximadamente algumas centenas de tickers distintos), tornando o procedimento viável.

Ao incluir os campos ano_inicial e ano_final, a tabela preserva a validade temporal de cada correspondência, permitindo que os dados fundamentalistas de uma companhia sejam vinculados corretamente ao ticker utilizado em cada período do backtest e evitando associações incorretas decorrentes de mudanças cadastrais ao longo dos anos.